# Allergy Scan — ViT 한국 음식 분류 학습

**모델:** `google/vit-base-patch16-224`  
**데이터:** AI Hub 한국 음식 150 클래스 (~150,000장)  
**환경:** Google Colab Free T4 (16GB)  
**예상 시간:** ~30분 (10 epochs, FP16, Batch 64)

---
### 실행 전 체크리스트
- [ ] 런타임 → T4 GPU 선택
- [ ] Drive에 `/allergydata/data/processed/` 업로드 완료
- [ ] Drive에 `/allergydata/data/label_ingredient_map.json` 업로드 완료

## 0. 패키지 설치

In [1]:
%%capture
!pip install transformers==4.44.0 datasets==2.21.0 timm faiss-cpu onnx onnxruntime accelerate

## 1. Google Drive 마운트

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/processed'
CKPT_DIR   = '/content/drive/MyDrive/allergydata/checkpoints'
MODEL_DIR  = '/content/drive/MyDrive/allergydata/models'

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# 클래스 목록 확인
classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'클래스 수: {len(classes)}')
print(classes[:10])

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/allergydata/data/processed'

In [4]:
import os
# 드라이브 안에 뭐가 있는지 확인
for item in os.listdir('/content/drive/MyDrive'):
    print(item)


제목 없는 스프레드시트 (1).gsheet
p-14_s.odt
p-14_s (1).gdoc
p-14_s.gdoc
프레젠테이션1.pptx
Google AI Studio
정해찬_2023270679_grits 설계서.txt
고려대학교(세종캠퍼스)_1팀_Quiziehub.pdf
정해찬서명.jpg
정혜규 깔따기.gsheet
소상공인 정책자금 상담 신청.gsheet
제목 없는 스프레드시트.gsheet
processed
allergydata


## 2. 데이터셋 준비

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import random

label2id = {cls: i for i, cls in enumerate(classes)}
id2label = {i: cls for i, cls in enumerate(classes)}

# ── 이미지 경로 수집 ──────────────────────────────────────
all_samples = []  # [(path, label_id), ...]
for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_samples.append((os.path.join(cls_dir, fname), label2id[cls]))

random.shuffle(all_samples)
split = int(len(all_samples) * 0.9)
train_samples = all_samples[:split]
val_samples   = all_samples[split:]
print(f'train: {len(train_samples):,}  val: {len(val_samples):,}')

In [ ]:
# ── MixUp 구현 ────────────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = torch.distributions.Beta(alpha, alpha).sample().item()
    else:
        lam = 1.0
    idx = torch.randperm(x.size(0))
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


# ── Dataset ───────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class FoodDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        pixel_values = self.transform(img)
        return {'pixel_values': pixel_values, 'labels': torch.tensor(label)}


train_ds = FoodDataset(train_samples, train_tf)
val_ds   = FoodDataset(val_samples,   val_tf)
print('Dataset 준비 완료')

## 3. 모델 로드

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor

MODEL_NAME = 'google/vit-base-patch16-224'

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'클래스 수: {model.config.num_labels}')

## 4. 학습 설정

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
import evaluate

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir=CKPT_DIR,

    # 에폭 & 배치
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,

    # 옵티마이저
    learning_rate=2e-5,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,

    # FP16 (T4 필수)
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,

    # 체크포인트 (Drive 저장)
    save_strategy='epoch',
    save_total_limit=3,
    evaluation_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',

    # 로깅
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Trainer 준비 완료')

## 5. 학습 시작

In [ ]:
# 이전 체크포인트에서 이어서 학습하려면:
# trainer.train(resume_from_checkpoint=True)

trainer.train()

# 최종 모델 저장
trainer.save_model(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)
print(f'모델 저장 완료: {MODEL_DIR}')

## 6. 검증 결과 확인

In [ ]:
results = trainer.evaluate()
print(f"Val Accuracy : {results['eval_accuracy']:.4f}")
print(f"Val Loss     : {results['eval_loss']:.4f}")

# 목표: Val Acc > 0.75
if results['eval_accuracy'] >= 0.75:
    print('✅ 목표 달성 (>75%)')
else:
    print('⚠️  목표 미달 — 클래스 수 줄이기 or 데이터 품질 재검토 필요')

## 7. FAISS 인덱스 구축

In [ ]:
import faiss
import torch
import numpy as np
from torch.utils.data import DataLoader

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 클래스별 임베딩 centroid 계산
print('임베딩 추출 중...')
class_embeddings = {}

for cls_name in classes:
    cls_id  = label2id[cls_name]
    samples = [(p, l) for p, l in val_samples if l == cls_id][:50]  # 클래스당 최대 50장
    if not samples:
        continue

    ds  = FoodDataset(samples, val_tf)
    dl  = DataLoader(ds, batch_size=32)
    embs = []

    with torch.no_grad():
        for batch in dl:
            px = batch['pixel_values'].to(device)
            outputs = model.vit(pixel_values=px)
            embs.append(outputs.last_hidden_state[:, 0].cpu().numpy())  # CLS 토큰

    class_embeddings[cls_name] = np.concatenate(embs).mean(axis=0)  # centroid

# FAISS IndexFlatL2 구축 (200 클래스 수준, IVF 불필요)
dim        = 768
index      = faiss.IndexFlatL2(dim)
index_ids  = list(class_embeddings.keys())
vectors    = np.array([class_embeddings[k] for k in index_ids], dtype='float32')
index.add(vectors)

# 저장
faiss.write_index(index, f'{MODEL_DIR}/faiss_index.bin')
import json
with open(f'{MODEL_DIR}/faiss_labels.json', 'w', encoding='utf-8') as f:
    json.dump(index_ids, f, ensure_ascii=False)

print(f'FAISS 인덱스 저장 완료 ({len(index_ids)} 클래스, dim={dim})')

## 8. ONNX 변환 (INT8 양자화)

In [ ]:
import torch
from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_fp32_path = f'{MODEL_DIR}/model_fp32.onnx'
onnx_int8_path = f'{MODEL_DIR}/model_int8.onnx'

# FP32 ONNX 내보내기
dummy = torch.randn(1, 3, 224, 224).to(device)
model.eval()

torch.onnx.export(
    model,
    {'pixel_values': dummy},
    onnx_fp32_path,
    input_names=['pixel_values'],
    output_names=['logits'],
    dynamic_axes={'pixel_values': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=14,
)
print(f'FP32 ONNX 저장: {onnx_fp32_path}')

# INT8 동적 양자화
quantize_dynamic(
    onnx_fp32_path,
    onnx_int8_path,
    weight_type=QuantType.QInt8,
)
print(f'INT8 ONNX 저장: {onnx_int8_path}')

# 파일 크기 비교
fp32_mb = os.path.getsize(onnx_fp32_path) / 1e6
int8_mb = os.path.getsize(onnx_int8_path) / 1e6
print(f'FP32: {fp32_mb:.1f} MB  →  INT8: {int8_mb:.1f} MB  ({int8_mb/fp32_mb*100:.0f}%)')

## 9. 추론 테스트

In [ ]:
import onnxruntime as ort
import json
import numpy as np
from PIL import Image
from torchvision import transforms

# ONNX 세션
sess = ort.InferenceSession(onnx_int8_path, providers=['CPUExecutionProvider'])

# FAISS 인덱스 로드
index = faiss.read_index(f'{MODEL_DIR}/faiss_index.bin')
with open(f'{MODEL_DIR}/faiss_labels.json', encoding='utf-8') as f:
    index_labels = json.load(f)

def predict(img_path: str, top_k: int = 3):
    img = Image.open(img_path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).numpy()

    logits = sess.run(None, {'pixel_values': tensor})[0][0]
    probs  = np.exp(logits) / np.exp(logits).sum()
    top_ids = np.argsort(probs)[::-1][:top_k]

    return [
        {'label': id2label[i], 'score': float(probs[i])}
        for i in top_ids
    ]

# 샘플 이미지로 테스트
sample_path, sample_label = val_samples[0]
results = predict(sample_path)
print(f'정답: {id2label[sample_label]}')
print('예측 top-3:')
for r in results:
    print(f"  {r['label']:20s}  {r['score']:.4f}")